XGBREGRESSION-(XGBOOST)

IMPORTS

In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
from xgboost import XGBRegressor

In [25]:
df = pd.read_csv('cleaned_dataset.csv')

FEATURES & TARGET

In [26]:
X = df.drop(columns=['Calories'])
# HAN4EL EL COLUMNS MEN EL FEATURES 34AN TEB2A EL TARGET

y = df['Calories']
# CALORIES IS TARGET

ENCODING

In [27]:
# ONE HOT ENCODING FOR DAY OF THE WEEK
X_XGB = pd.get_dummies(X, columns=['DayOfWeek'], drop_first=True, dtype=int)
print("ENCODING SHAPE = ", X_XGB.shape)
print("DAY OF THE WEEK COLUMNS", list(X_XGB.columns[9:]))
print("ALL COLUMNS", list(X_XGB.columns))
#CHANGED DAY OF WEEK TO SEPARATED DAYS

ENCODING SHAPE =  (2532, 15)
DAY OF THE WEEK COLUMNS ['DayOfWeek_Monday', 'DayOfWeek_Saturday', 'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday', 'DayOfWeek_Wednesday']
ALL COLUMNS ['StepTotal', 'TotalIntensity', 'AvgHeartRate', 'MaxHeartRate', 'MinHeartRate', 'SleepMinutes', 'WeightKg', 'BMI', 'Hour', 'DayOfWeek_Monday', 'DayOfWeek_Saturday', 'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday', 'DayOfWeek_Wednesday']


TRAIN & TEST & GRID SEARCH

In [28]:
# MAKE TRAIN = 80%  MAKE TEST = 20%
X_train, X_test, y_train, y_test = train_test_split(
    X_XGB, y, test_size=0.2, random_state=0
)
xgb_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('xgb', XGBRegressor(random_state=0, n_jobs=-1))
])

xgb_param_grid = {
    'xgb__n_estimators': [150, 250],
    'xgb__learning_rate': [0.05, 0.1],
    'xgb__max_depth': [4, 6, 8]
}

xgb_grid = GridSearchCV(
    xgb_pipe,
    param_grid=xgb_param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

xgb_grid.fit(X_train, y_train)
best_xgb = xgb_grid.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

BEFORE & AFTER GRID

In [29]:
xgb_pipe.fit(X_train, y_train)
y_train_base = xgb_pipe.predict(X_train)
y_test_base = xgb_pipe.predict(X_test)

print("=== Before GridSearchCV ===")
print(f"Train R2 : {r2_score(y_train, y_train_base):.4f}")
print(f"Test  R2 : {r2_score(y_test, y_test_base):.4f}")
print(f"MAE      : {mean_absolute_error(y_test, y_test_base):.2f}")
print(f"MSE      : {mean_squared_error(y_test, y_test_base):.2f}")
print(f"RMSE     : {root_mean_squared_error(y_test, y_test_base):.2f}")

y_train_best = best_xgb.predict(X_train)
print("\n=== After GridSearchCV ===")
print(f"Train R2 : {r2_score(y_train, y_train_best):.4f}")
print(f"Test  R2 : {r2_score(y_test, y_pred_xgb):.4f}")
print(f"MAE      : {mean_absolute_error(y_test, y_pred_xgb):.2f}")
print(f"MSE      : {mean_squared_error(y_test, y_pred_xgb):.2f}")
print(f"RMSE     : {root_mean_squared_error(y_test, y_pred_xgb):.2f}")

=== Before GridSearchCV ===
Train R2 : 0.9983
Test  R2 : 0.9192
MAE      : 13.26
MSE      : 500.19
RMSE     : 22.36

=== After GridSearchCV ===
Train R2 : 0.9735
Test  R2 : 0.9284
MAE      : 13.14
MSE      : 442.88
RMSE     : 21.04


PREDICTION

In [31]:
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = root_mean_squared_error(y_test, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print("Best XGB Params:", xgb_grid.best_params_)
print("--------------")
print("R2 score")
print(f"{r2_xgb:.4f}")
print("--------------")
print("MAE")
print(f"{mae_xgb:.2f}")
print("--------------")
print("MSE")
print(f"{mse_xgb:.2f}")
print("--------------")
print("RMSE")
print(f"{rmse_xgb:.2f}")
print("--------------")

Best XGB Params: {'xgb__learning_rate': 0.05, 'xgb__max_depth': 4, 'xgb__n_estimators': 250}
--------------
R2 score
0.9284
--------------
MAE
13.14
--------------
MSE
442.88
--------------
RMSE
21.04
--------------


In [32]:
importances_xgb = pd.Series(
    best_xgb.named_steps['xgb'].feature_importances_,
    index=X_XGB.columns
).sort_values(ascending=False)
print("\nTop 8 Features (XGB):\n", importances_xgb.head(8).round(4))


Top 8 Features (XGB):
 TotalIntensity    0.6430
StepTotal         0.1217
AvgHeartRate      0.0718
WeightKg          0.0482
BMI               0.0259
MaxHeartRate      0.0248
SleepMinutes      0.0169
MinHeartRate      0.0157
dtype: float32


In [38]:

best_xgb_params = xgb_grid.best_params_
test_sizes = [0.1, 0.2, 0.3, 0.4]
xgb_tuned = None

print("=== XGBoost across test_sizes ===")
for size in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(
        X_XGB, y, test_size=size, random_state=0
    )

    xgb_tuned = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('xgb', XGBRegressor(
            n_estimators=best_xgb_params['xgb__n_estimators'],
            learning_rate=best_xgb_params['xgb__learning_rate'],
            max_depth=best_xgb_params['xgb__max_depth'],
            random_state=0,
            n_jobs=-1 ,
        ))
    ])

    xgb_tuned.fit(X_train, y_train)
    y_pred = xgb_tuned.predict(X_test)
    
    print(f"--- Test Size: {size * 100:.0f}% ---")
    print("-------------------")

    print("-------------------")
    print("R2 SCORE")
    print(f"{r2_score(y_test, y_pred):.4f}")
    print("-------------------")
    print("MAE")
    print(f"{mean_absolute_error(y_test, y_pred):.2f}")
    print("-------------------")
    print("MSE")
    print(f"{mean_squared_error(y_test, y_pred):.2f}")
    print("-------------------")
    print("RMSE")
    print(f"{root_mean_squared_error(y_test, y_pred):.2f}")
    print("-------------------")

=== XGBoost across test_sizes ===
--- Test Size: 10% ---
-------------------
-------------------
R2 SCORE
0.9176
-------------------
MAE
14.44
-------------------
MSE
568.77
-------------------
RMSE
23.85
-------------------
--- Test Size: 20% ---
-------------------
-------------------
R2 SCORE
0.9284
-------------------
MAE
13.14
-------------------
MSE
442.88
-------------------
RMSE
21.04
-------------------
--- Test Size: 30% ---
-------------------
-------------------
R2 SCORE
0.9210
-------------------
MAE
12.87
-------------------
MSE
449.48
-------------------
RMSE
21.20
-------------------
--- Test Size: 40% ---
-------------------
-------------------
R2 SCORE
0.9145
-------------------
MAE
13.53
-------------------
MSE
513.01
-------------------
RMSE
22.65
-------------------
